In [1]:
"""
Post-processing of the GridMET climatology data
~ Build ERC inputs for FSPro by CO Pyrome
"""

import os, sys

from pathlib import Path
from fb_tools.weather import (
    load_gridmet_csv,
    build_historic_erc_arrays,
    build_erc_stats,
    build_erc_classes,
    build_current_erc_values,
)

# use the current working directory
projdir = Path.cwd().parents[1] # moves up two, outside code directory
print(f"Project directory set to: {projdir}")

# environ vars
proj_crs = 26913  # NAD83 UTM Zone 13N

Project directory set to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/fire_modeling/FM_PythonWrapper


## Post-processing: Build per-pyrome FSPro weather inputs from exported CSV

Once the GEE export completes and the CSV is downloaded to ``data/weather/``,
run the cells below to build FSPro-ready ERC arrays and ERC class tables.

**Expected CSV path:** ``data/weather/gridmet_erc_pyromes.csv``

In [2]:
# --- Paths
csv_path = projdir / 'data' / 'tabular' / 'raw' / 'weather' / 'gridmet_clima_co_pyromes.csv'
erc_cache_dir = projdir / 'data' / 'weather' / 'pyrome_erc'
erc_cache_dir.mkdir(parents=True, exist_ok=True)

# --- Load and inspect
clim = load_gridmet_csv(csv_path)
print(f"Loaded {len(clim):,} rows × {clim.shape[1]} columns")
print(clim.dtypes)
clim.head()

  [load_gridmet_csv] 28,890 rows, 9 pyromes, years 2008–2022
Loaded 28,890 rows × 11 columns
system:index            object
pyrome                   int64
date            datetime64[ns]
doy                      int64
erc                    float64
fm100                  float64
fm1000                 float64
rmin                   float64
year                     int64
.geo                    object
tmmx_f                 float64
dtype: object


,system:index,pyrome,date,doy,erc,fm100,fm1000,rmin,year,.geo,tmmx_f
0,0_00000000000000000029,42,2008-04-01,92,65.185247,8.108735,9.765048,19.814494,2008,"{""type"":""MultiPoint"",""coordinates"":[]}",56.764894
1,0_0000000000000000002a,43,2008-04-01,92,37.808192,13.307531,14.425369,25.121087,2008,"{""type"":""MultiPoint"",""coordinates"":[]}",43.635754
2,0_0000000000000000002c,45,2008-04-01,92,26.389515,15.500090,16.624220,33.752734,2008,"{""type"":""MultiPoint"",""coordinates"":[]}",36.787494
3,0_0000000000000000002d,46,2008-04-01,92,32.910168,14.474093,15.237802,34.067851,2008,"{""type"":""MultiPoint"",""coordinates"":[]}",36.332295
4,0_0000000000000000002e,47,2008-04-01,92,57.385668,8.692565,11.158137,22.515070,2008,"{""type"":""MultiPoint"",""coordinates"":[]}",49.697530


In [3]:
# --- Clean up some of the columns
clim.drop(columns=['system:index','.geo'], inplace=True)
print(clim.columns)

Index(['pyrome', 'date', 'doy', 'erc', 'fm100', 'fm1000', 'rmin', 'year',
       'tmmx_f'],
      dtype='object')


In [4]:
# --- Build historic ERC arrays (15yr × 214day) per pyrome + write JSON cache
historic = build_historic_erc_arrays(clim, out_dir=erc_cache_dir)
print(f"\nPyromes processed: {len(historic)}")
for pid, arr in list(historic.items())[:3]:
    print(f"  Pyrome {pid}: shape {arr.shape}  (years × DOYs)")

# --- ERC stats: average and std-dev per DOY across years
stats = build_erc_stats(historic)
print(f"\nERC stats computed for {len(stats)} pyromes")
print(f"  avg shape: {list(stats.values())[0]['avg'].shape}")

# --- ERC classes: 5-row table with FM values per pyrome
classes = build_erc_classes(clim)
print(f"\nERC classes built for {len(classes)} pyromes")
for pid, cls in list(classes.items())[:2]:
    print(f"\n  Pyrome {pid}:")
    print(f"  {'lower':>6} {'upper':>6} {'fm1':>5} {'fm10':>5} "
          f"{'fm100':>6} {'fm_herb':>7} {'fm_woody':>8} {'spot_d':>7} "
          f"{'spot_p':>7} {'spot2':>6}")
    for row in cls:
        print(f"  {row[0]:>6.0f} {row[1]:>6.0f} {row[2]:>5.1f} "
              f"{row[3]:>5.1f} {row[4]:>6.1f} {row[5]:>7.1f} {row[6]:>8.1f} "
              f"{row[7]:>7.0f} {row[8]:>7.2f} {row[9]:>6.0f}")

  [gridmet] Wrote pyrome_42_gridmet.json
  [gridmet] Wrote pyrome_43_gridmet.json
  [gridmet] Wrote pyrome_45_gridmet.json
  [gridmet] Wrote pyrome_46_gridmet.json
  [gridmet] Wrote pyrome_47_gridmet.json
  [gridmet] Wrote pyrome_52_gridmet.json
  [gridmet] Wrote pyrome_53_gridmet.json
  [gridmet] Wrote pyrome_56_gridmet.json
  [gridmet] Wrote pyrome_128_gridmet.json
  [build_historic_erc_arrays] 9 pyromes → arrays shape (15, 214)

Pyromes processed: 9
  Pyrome 42: shape (15, 214)  (years × DOYs)
  Pyrome 43: shape (15, 214)  (years × DOYs)
  Pyrome 45: shape (15, 214)  (years × DOYs)

ERC stats computed for 9 pyromes
  avg shape: (214,)
  [build_erc_classes] 9 pyromes × 5 ERC classes

ERC classes built for 9 pyromes

  Pyrome 42:
   lower  upper   fm1  fm10  fm100 fm_herb fm_woody  spot_d  spot_p  spot2
      86    106   2.1   2.6    4.3    10.0     20.0     360    0.15      0
      76     86   3.1   3.8    5.8    10.8     20.8     300    0.10      0
      68     76   3.7   4.6    7.1